# Estrazione delle quadruple (aspect-opinion-category-sentiment) con GLINER2 su RESTAURANT-ACOS

In [2]:
import pandas as pd
import os
from gliner2 import GLiNER2
from tqdm import tqdm
import torch
import pandas as pd
from seqeval.metrics import classification_report as seqeval_report
from sklearn.metrics import classification_report as sklearn_report

# --- 1. CONFIGURAZIONE DEL DEVICE ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f" GPU Trovata: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(" Acceleratore Apple Metal (MPS) Trovato")
else:
    device = torch.device("cpu")
    print(" Nessuna GPU trovata. L'addestramento sarà lento.")

print("Librerie caricate.")

 Acceleratore Apple Metal (MPS) Trovato
Librerie caricate.


In [5]:
# 1. CARICAMENTO DEL DATASET PULITO (Usiamo il Test Set dei Ristoranti)
percorso_test = "data_parsing/test_rest_parsed.pkl"
print(f"Caricamento dataset: {percorso_test}")
df_test = pd.read_pickle(percorso_test)

# 2. ESTRAZIONE DINAMICA DELLE CATEGORIE
# Invece di scriverle a mano, le estraiamo direttamente dalla ground truth del dataset
categorie_uniche = set()
for quads in df_test['parsed_quadruples']:
    for q in quads:
        categorie_uniche.add(q['category_aspect'])

lista_categorie = "|".join(sorted(list(categorie_uniche)))
print(f"Categorie ACOS trovate ({len(categorie_uniche)}): {lista_categorie}\n")

# 3. CARICAMENTO DEL MODELLO GLiNER 2
print("Inizializzazione di GLiNER 2 (Large) in corso...")
extractor = GLiNER2.from_pretrained("fastino/gliner2-large-v1")

# 4. DEFINIZIONE DELLO SCHEMA RAFFINATO (Prompt Engineering Avanzato)
schema_acos = {
    "acos_quadruples": [
        "aspect_term::str::Extract ONLY the core noun or noun phrase representing the specific item, food, or service being reviewed (e.g., 'pizza', 'green tea creme brulee', 'staff'). DO NOT include verbs, articles, or punctuation. If the aspect is implicit, missing, or referred to as 'it', you MUST output exactly 'NULL'.",
        
        "opinion_term::str::Extract ONLY the specific exact word or short phrase that expresses the sentiment (e.g., 'delicious', 'rude', 'must'). STRICTLY DO NOT include articles (like 'a', 'the'), verbs (like 'is', 'was'), or punctuation marks (like '!'). If implicit, output exactly 'NULL'.",
        
        f"category::[{lista_categorie}]::str::The specific category of the aspect.",
        
        "sentiment::[Negative|Neutral|Positive|Conflict]::str::The sentiment polarity of the opinion."
    ]
}

# Mappatura per riportare il sentimento testuale al formato numerico di ACOS
mappa_sentimento = {"Negative": 0, "Neutral": 1, "Positive": 2, "Conflict": 3}

risultati_zero_shot = []

print(f"Avvio estrazione Zero-Shot su {len(df_test)} recensioni...")

# 5. CICLO DI ESTRAZIONE
for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    testo_sporco = row['review_text']
    # RICUCIAMO LE PAROLE SPEZZATE DAI CANCELLETTI!
    testo_pulito = testo_sporco.replace(" ##", "")
    quadruple_reali = row['parsed_quadruples'] # Ground Truth
    
    try:
        # Estrazione strutturata
        risultato = extractor.extract_json(testo_pulito, schema_acos)
        quadruple_grezze = risultato.get("acos_quadruples", [])
        
        quadruple_pulite = []
        
        # Gestione dei NULL e formattazione
        for quad in quadruple_grezze:
            # Rete di sicurezza: se omette la chiave o scrive null, forziamo "NULL"
            aspect = str(quad.get('aspect_term', 'NULL')).strip()
            if not aspect or aspect.lower() == 'null': aspect = 'NULL'
                
            opinion = str(quad.get('opinion_term', 'NULL')).strip()
            if not opinion or opinion.lower() == 'null': opinion = 'NULL'
                
            category = quad.get('category', 'Sconosciuta')
            sent_text = quad.get('sentiment', 'Neutral')
            
            # Convertiamo il sentimento in numero
            sent_num = mappa_sentimento.get(sent_text, 1) # Default 1 (Neutral) se sbaglia
            
            quadruple_pulite.append({
                'aspect_term': aspect,
                'opinion_term': opinion,
                'category_aspect': category,
                'sentiment': sent_num
            })
            
        risultati_zero_shot.append({
            'review_text': testo_pulito,
            'gold_quadruples': quadruple_reali,
            'pred_quadruples': quadruple_pulite
        })
        
    except Exception as e:
        print(f"Errore alla riga {index}: {e}")
        risultati_zero_shot.append({
            'review_text': testo_pulito,
            'gold_quadruples': quadruple_reali,
            'pred_quadruples': []
        })

# 6. SALVATAGGIO DEI RISULTATI
df_predizioni = pd.DataFrame(risultati_zero_shot)
os.makedirs("risultati_gliner", exist_ok=True)
percorso_salvataggio = "risultati_gliner/test_rest_predictions.pkl"
df_predizioni.to_pickle(percorso_salvataggio)

print(f"\n Finito! Risultati salvati in '{percorso_salvataggio}'")

# Stampiamo un esempio per vedere come si è comportato!
print("\n--- ESEMPIO DI ESTRAZIONE ZERO-SHOT ---")
esempio = df_predizioni.iloc[3]
print(f"Testo: {esempio['review_text']}")
print(f"Predizioni GLiNER: {esempio['pred_quadruples']}")
print(f"Quadruple Reali  : {esempio['gold_quadruples']}")

Caricamento dataset: data_parsing/test_rest_parsed.pkl
Categorie ACOS trovate (12): AMBIENCE#GENERAL|DRINKS#PRICES|DRINKS#QUALITY|DRINKS#STYLE_OPTIONS|FOOD#PRICES|FOOD#QUALITY|FOOD#STYLE_OPTIONS|LOCATION#GENERAL|RESTAURANT#GENERAL|RESTAURANT#MISCELLANEOUS|RESTAURANT#PRICES|SERVICE#GENERAL

Inizializzazione di GLiNER 2 (Large) in corso...


You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-large
Counting layer     : count_lstm
Token pooling      : first
Avvio estrazione Zero-Shot su 583 recensioni...


100%|██████████| 583/583 [07:56<00:00,  1.22it/s]


 Finito! Risultati salvati in 'risultati_gliner/test_rest_predictions.pkl'

--- ESEMPIO DI ESTRAZIONE ZERO-SHOT ---
Testo: green tea creme brulee is a must !
Predizioni GLiNER: [{'aspect_term': 'green tea creme brulee', 'opinion_term': 'must', 'category_aspect': None, 'sentiment': 2}]
Quadruple Reali  : [{'span_A': (0, 7), 'span_B': (9, 10), 'category_aspect': 'FOOD#QUALITY', 'sentiment': 2}]


In [6]:
# 1. Carichiamo i risultati che hai appena generato
percorso_risultati = "risultati_gliner/test_rest_predictions.pkl"
df_predizioni = pd.read_pickle(percorso_risultati)

true_quads = []
pred_quads = []

print(" Allineamento formati (Spans -> Testo) in corso...")

for index, row in df_predizioni.iterrows():
    testo = row['review_text']
    # ACOS usa la tokenizzazione basata sugli spazi per i suoi indici
    token_list = testo.split() 
    
    # --- COSTRUZIONE DELLA GROUND TRUTH (TRUE SET) ---
    t_set = set()
    for q_true in row['gold_quadruples']:
        # Estrazione Aspetto (Ground Truth)
        span_a = q_true.get('span_A', (-1, -1))
        if span_a == (-1, -1) or span_a == [-1, -1]:
            aspect_text = "null"
        else:
            # Uniamo i token e RIMUIOVIAMO I CANCELLETTI per fare il match con GLiNER
            aspect_text = " ".join(token_list[span_a[0]:span_a[1]]).replace(" ##", "").lower()

        # Estrazione Opinione (Ground Truth)
        span_b = q_true.get('span_B', (-1, -1))
        if span_b == (-1, -1) or span_b == [-1, -1]:
            opinion_text = "null"
        else:
            # Stessa cosa per le opinioni!
            opinion_text = " ".join(token_list[span_b[0]:span_b[1]]).replace(" ##", "").lower()

        category = str(q_true['category_aspect']).lower()
        sentiment = int(q_true['sentiment'])
        
        t_set.add((aspect_text, opinion_text, category, sentiment))
    
    true_quads.append(t_set)

    # --- COSTRUZIONE DELLE PREDIZIONI GLiNER (PRED SET) ---
    p_set = set()
    for q_pred in row['pred_quadruples']:
        aspect_text = str(q_pred['aspect_term']).lower().strip()
        opinion_text = str(q_pred['opinion_term']).lower().strip()
        category = str(q_pred['category_aspect']).lower().strip()
        sentiment = int(q_pred['sentiment'])
        
        p_set.add((aspect_text, opinion_text, category, sentiment))
        
    pred_quads.append(p_set)

# --- LE TUE METRICHE ESATTE ---
total_pred = sum(len(p) for p in pred_quads)
total_true = sum(len(t) for t in true_quads)
correct = sum(len(p_set.intersection(t_set)) for p_set, t_set in zip(pred_quads, true_quads))

precision = correct / total_pred if total_pred > 0 else 0
recall = correct / total_true if total_true > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("\n" + "="*50)
print("RISULTATI FINALI EXACT MATCH ACOS (GLiNER ZERO-SHOT)")
print("="*50)
print(f"Quadruple Reali totali:    {total_true}")
print(f"Quadruple Predette totali: {total_pred}")
print(f"Quadruple Esatte:          {correct}")
print("-" * 50)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("="*50)

 Allineamento formati (Spans -> Testo) in corso...

RISULTATI FINALI EXACT MATCH ACOS (GLiNER ZERO-SHOT)
Quadruple Reali totali:    907
Quadruple Predette totali: 608
Quadruple Esatte:          64
--------------------------------------------------
Precision: 0.1053
Recall:    0.0706
F1-Score:  0.0845


In [7]:
print("\n" + "="*70)
print("🔍 ANALISI DEGLI ERRORI: ESEMPI DI 'PIGRIZIA' O CONFINI SBAGLIATI")
print("="*70)

contatore = 0
limite_esempi = 5 # Mostriamo 5 casi studio perfetti per la tesi

for i in range(len(df_predizioni)):
    t_set = true_quads[i]
    p_set = pred_quads[i]
    
    # Cerchiamo casi in cui la Ground Truth ha PIÙ quadruple di quelle predette (Bassa Recall)
    # OPPURE casi in cui l'intersezione è zero (Nessun Exact Match)
    intersezione = p_set.intersection(t_set)
    
    if (len(t_set) > len(p_set) or len(intersezione) == 0) and len(t_set) > 0:
        
        testo = df_predizioni.iloc[i]['review_text']
        
        print(f"\n📝 TESTO: {testo}")
        print(f"✅ GROUND TRUTH ({len(t_set)} quadruple reali):")
        for q in t_set:
            print(f"   -> {q}")
            
        print(f"🤖 PREDETTE ({len(p_set)} quadruple trovate):")
        if len(p_set) == 0:
            print("   -> [Vuoto! Il modello non ha trovato nulla o è andato in tilt]")
        else:
            for q in p_set:
                # Mettiamo una spunta a quelle (se ci sono) che ha azzeccato per caso
                status = "✔️ ESATTA" if q in intersezione else "❌ SBAGLIATA"
                print(f"   -> {q} [{status}]")
        
        print("-" * 70)
        
        contatore += 1
        if contatore >= limite_esempi:
            break


🔍 ANALISI DEGLI ERRORI: ESEMPI DI 'PIGRIZIA' O CONFINI SBAGLIATI

📝 TESTO: yum !
✅ GROUND TRUTH (1 quadruple reali):
   -> ('null', 'yum !', 'food#quality', 2)
🤖 PREDETTE (1 quadruple trovate):
   -> ('none', 'yum !', 'food#quality', 2) [❌ SBAGLIATA]
----------------------------------------------------------------------

📝 TESTO: serves really good sushi .
✅ GROUND TRUTH (1 quadruple reali):
   -> ('sushi .', 'good', 'food#quality', 2)
🤖 PREDETTE (1 quadruple trovate):
   -> ('sushi', 'really good', 'food#quality', 2) [❌ SBAGLIATA]
----------------------------------------------------------------------

📝 TESTO: not the biggest portions but adequate .
✅ GROUND TRUTH (2 quadruple reali):
   -> ('portions', 'not the biggest', 'food#style_options', 1)
   -> ('portions', 'adequate', 'food#style_options', 1)
🤖 PREDETTE (1 quadruple trovate):
   -> ('biggest portions', 'adequate', 'food#quality', 1) [❌ SBAGLIATA]
----------------------------------------------------------------------

📝 TESTO

In [4]:
# 1. Carica le predizioni
percorso_risultati = "risultati_gliner/test_rest_predictions.pkl"
df_predizioni = pd.read_pickle(percorso_risultati)

y_true_bio = []
y_pred_bio = []

print("🔄 Conversione del testo e degli span in formato BIO in corso...")

for index, row in df_predizioni.iterrows():
    # ACOS usa gli spazi per dividere i token
    tokens = row['review_text'].split()
    
    # Inizializziamo tutte le parole come 'O' (Outside)
    true_tags = ['O'] * len(tokens)
    pred_tags = ['O'] * len(tokens)
    
    # --- COSTRUZIONE DELLA GROUND TRUTH IN BIO ---
    for q_true in row['gold_quadruples']:
        # Mappatura Aspetto
        span_a = q_true.get('span_A', (-1, -1))
        if span_a != (-1, -1) and span_a != [-1, -1]:
            start, end = span_a
            if start < len(tokens):
                true_tags[start] = 'B-ASP'
                for i in range(start + 1, min(end, len(tokens))):
                    true_tags[i] = 'I-ASP'
        
        # Mappatura Opinione
        span_b = q_true.get('span_B', (-1, -1))
        if span_b != (-1, -1) and span_b != [-1, -1]:
            start, end = span_b
            if start < len(tokens):
                true_tags[start] = 'B-OPI'
                for i in range(start + 1, min(end, len(tokens))):
                    true_tags[i] = 'I-OPI'
                    
    # --- COSTRUZIONE DELLE PREDIZIONI GLiNER IN BIO ---
    for q_pred in row['pred_quadruples']:
        aspect_text = str(q_pred['aspect_term']).strip()
        opinion_text = str(q_pred['opinion_term']).strip()
        
        # Funzione per trovare gli indici (spans) del testo estratto da GLiNER
        def trova_indici(testo_estratto, lista_token):
            if not testo_estratto or testo_estratto.lower() in ['null', 'none']: return -1, -1
            sub_tokens = testo_estratto.split()
            n = len(sub_tokens)
            
            # 1. Prova a cercare un match esatto (case insensitive)
            for i in range(len(lista_token) - n + 1):
                if [t.lower() for t in lista_token[i:i+n]] == [t.lower() for t in sub_tokens]:
                    return i, i+n
                    
            # 2. Rete di sicurezza: prova a ignorare i "##" nella ricerca
            clean_main = [t.replace("##", "").lower() for t in lista_token]
            for i in range(len(clean_main) - n + 1):
                if clean_main[i:i+n] == [t.lower() for t in sub_tokens]:
                    return i, i+n
            return -1, -1
        
        # Riapplichiamo il BIO alle predizioni
        start_a, end_a = trova_indici(aspect_text, tokens)
        if start_a != -1:
            pred_tags[start_a] = 'B-ASP'
            for i in range(start_a + 1, end_a):
                pred_tags[i] = 'I-ASP'
                
        start_o, end_o = trova_indici(opinion_text, tokens)
        if start_o != -1:
            pred_tags[start_o] = 'B-OPI'
            for i in range(start_o + 1, end_o):
                pred_tags[i] = 'I-OPI'
                
    y_true_bio.append(true_tags)
    y_pred_bio.append(pred_tags)

# =====================================================================
# CALCOLO DELLE DUE METRICHE RICHIESTE DAL PROFESSORE
# =====================================================================

print("\n" + "="*60)
print("1. ENTITY-LEVEL EVALUATION (Libreria 'seqeval')")
print("Valuta il blocco intero. L'entità deve combaciare al 100%.")
print("="*60)
print(seqeval_report(y_true_bio, y_pred_bio))

# Per il token level, dobbiamo "appiattire" le liste di liste in una singola lista gigante
y_true_flat = [tag for seq in y_true_bio for tag in seq]
y_pred_flat = [tag for seq in y_pred_bio for tag in seq]

print("\n" + "="*60)
print("2. TOKEN-LEVEL EVALUATION (Libreria 'sklearn')")
print("Valuta ogni singola parola. Assegna credito parziale.")
print("="*60)
# Filtriamo l'etichetta 'O' (Outside) per concentrarci solo su Aspetti e Opinioni
etichette_da_valutare = ['B-ASP', 'I-ASP', 'B-OPI', 'I-OPI']
print(sklearn_report(y_true_flat, y_pred_flat, labels=etichette_da_valutare))

# =====================================================================
# CALCOLO PUNTEGGI PARZIALI: CATEGORIA E SENTIMENTO
# =====================================================================

tot_true_cat = 0
tot_pred_cat = 0
correct_cat = 0

tot_true_sent = 0
tot_pred_sent = 0
correct_sent = 0

for index, row in df_predizioni.iterrows():
    # Estraiamo SOLO le categorie reali e predette per questa frase
    t_cat_set = set([str(q['category_aspect']).lower().strip() for q in row['gold_quadruples']])
    p_cat_set = set([str(q['category_aspect']).lower().strip() for q in row['pred_quadruples']])
    
    tot_true_cat += len(t_cat_set)
    tot_pred_cat += len(p_cat_set)
    correct_cat += len(p_cat_set.intersection(t_cat_set))
    
    # Estraiamo SOLO i sentimenti reali e predetti per questa frase
    t_sent_set = set([int(q['sentiment']) for q in row['gold_quadruples']])
    p_sent_set = set([int(q['sentiment']) for q in row['pred_quadruples']])
    
    tot_true_sent += len(t_sent_set)
    tot_pred_sent += len(p_sent_set)
    correct_sent += len(p_sent_set.intersection(t_sent_set))

# Calcolo metriche Categoria
p_cat = correct_cat / tot_pred_cat if tot_pred_cat > 0 else 0
r_cat = correct_cat / tot_true_cat if tot_true_cat > 0 else 0
f1_cat = 2 * p_cat * r_cat / (p_cat + r_cat) if (p_cat + r_cat) > 0 else 0

# Calcolo metriche Sentimento
p_sent = correct_sent / tot_pred_sent if tot_pred_sent > 0 else 0
r_sent = correct_sent / tot_true_sent if tot_true_sent > 0 else 0
f1_sent = 2 * p_sent * r_sent / (p_sent + r_sent) if (p_sent + r_sent) > 0 else 0

print("\n" + "="*60)
print("3. CLASSIFICATION EVALUATION (Punteggi Parziali Astratti)")
print("="*60)

print(f"--- CATEGORY PREDICTION ---")
print(f"Precision: {p_cat:.4f}  |  Recall: {r_cat:.4f}  |  F1-Score: {f1_cat:.4f}")
print(f"(Categorie trovate: {correct_cat} su {tot_true_cat})")

print(f"\n--- SENTIMENT PREDICTION ---")
print(f"Precision: {p_sent:.4f}  |  Recall: {r_sent:.4f}  |  F1-Score: {f1_sent:.4f}")
print(f"(Sentimenti trovati: {correct_sent} su {tot_true_sent})")
print("="*60)

🔄 Conversione del testo e degli span in formato BIO in corso...

1. ENTITY-LEVEL EVALUATION (Libreria 'seqeval')
Valuta il blocco intero. L'entità deve combaciare al 100%.
              precision    recall  f1-score   support

         ASP       0.43      0.31      0.36       582
         OPI       0.42      0.29      0.34       605

   micro avg       0.42      0.30      0.35      1187
   macro avg       0.42      0.30      0.35      1187
weighted avg       0.42      0.30      0.35      1187


2. TOKEN-LEVEL EVALUATION (Libreria 'sklearn')
Valuta ogni singola parola. Assegna credito parziale.
              precision    recall  f1-score   support

       B-ASP       0.62      0.45      0.52       582
       I-ASP       0.58      0.18      0.28       487
       B-OPI       0.59      0.40      0.48       605
       I-OPI       0.15      0.25      0.18       195

   micro avg       0.49      0.34      0.40      1869
   macro avg       0.48      0.32      0.36      1869
weighted avg       